# PFE Navigator – Assistant intelligent basé sur RAG

## 1. Présentation du projet

PFE Navigator est un assistant intelligent qui répond aux questions
des étudiants concernant le Projet de Fin d'Études (PFE).

Il utilise une collection de documents PDF pour rechercher
les informations pertinentes et générer des réponses
en indiquant les sources utilisées.

## 2. Objectif

Construire un système RAG capable de :
- Lire des documents PDF.
- Extraire et nettoyer leur contenu.
- Découper les textes en morceaux (chunks).
- Retrouver les passages pertinents.
- Générer des réponses avec leurs sources.

In [1]:
import pypdf
import re
from pathlib import Path

print("Les bibliothèques sont prêtes !")

Les bibliothèques sont prêtes !


## 3. Chargement et extraction des documents PDF

Dans cette étape, nous allons parcourir le dossier documents,
ouvrir chaque fichier PDF et extraire son texte page par page.

Nous allons également supprimer les pieds de page commençant
par « ISIT – » afin d'éviter de répéter ces informations
dans les futurs chunks.

In [2]:
from pathlib import Path
from pypdf import PdfReader
import re

# Chemin vers le dossier contenant les PDF
documents_path = Path("../documents")

# Liste qui va contenir toutes les pages extraites
documents = []

# Parcourir tous les fichiers PDF
for pdf_file in documents_path.glob("*.pdf"):

    reader = PdfReader(pdf_file)

    print("Lecture du document :", pdf_file.name)

    # Parcourir chaque page
    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text:

            lines = text.splitlines()

            # Supprimer les pieds de page
            lines = [
                line for line in lines
                if not line.strip().startswith("ISIT –")
            ]

            clean_text = "\n".join(lines)
            clean_text = re.sub(r"\n{2,}", "\n", clean_text)

            documents.append({
                "text": clean_text.strip(),
                "source": pdf_file.name,
                "page": page_number
            })

print("\nExtraction terminée !")
print("Nombre de pages extraites :", len(documents))
print("Nombre de documents PDF :", len(set(d["source"] for d in documents)))

Lecture du document : 01_guide_pfe.pdf
Lecture du document : 02_reglement_stages.pdf
Lecture du document : 03_guide_redaction.pdf
Lecture du document : 04_guide_soutenance.pdf
Lecture du document : 05_calendrier_universitaire.pdf

Extraction terminée !
Nombre de pages extraites : 13
Nombre de documents PDF : 5


In [3]:
#Afficher un extrait pour vérifier
for doc in documents[:3]:

    print("Document :", doc["source"])
    print("Page :", doc["page"])
    print("\nContenu :")
    print(doc["text"][:500])
    print("\n" + "=" * 60)

Document : 01_guide_pfe.pdf
Page : 1

Contenu :
Page 1
Guide du Projet de Fin d'Études (PFE) – ISIT –
2026-2027
Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01
1. Définition et objectifs du PFE
Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ingénieur de l'ISIT, réalisée en
troisième année. Il vaut 30 crédits ECTS et se déroule en entreprise ou en laboratoire de
recherche. Son objectif est de mettre l'étudiant en situation professionnelle : analyser un besoin,
concevoir, développer, tester 

Document : 01_guide_pfe.pdf
Page : 2

Contenu :
Page 2
L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux
semaines.
7. Documents à fournir pendant le PFE
•
La convention de PFE, signée par les trois parties (étudiant, entreprise, ISIT) avant le
démarrage.
•
Un rapport de mi-parcours de 5 à 10 pages, à rendre avant le 15 avril 2027.
•
Le pré-rapport complet, à rendre avant le 24 mai 2027 pour relecture p

In [4]:
# Vérifier le début et la fin du texte de quelques pages

for doc in documents[:3]:

    print("Document :", doc["source"])
    print("Page :", doc["page"])

    print("\nDébut du texte :")
    print(doc["text"][:250])

    print("\nFin du texte :")
    print(doc["text"][-250:])

    print("\n" + "=" * 60)

Document : 01_guide_pfe.pdf
Page : 1

Début du texte :
Page 1
Guide du Projet de Fin d'Études (PFE) – ISIT –
2026-2027
Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01
1. Définition et objectifs du PFE
Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ing

Fin du texte :
e scientifique, relit le rapport et organise au moins 4 réunions de suivi.
•
L'encadrant professionnel est un ingénieur ou un cadre de la structure d'accueil, ayant au
moins 3 ans d'expérience. Il définit les tâches au quotidien et évalue l'étudiant.

Document : 01_guide_pfe.pdf
Page : 2

Début du texte :
Page 2
L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux
semaines.
7. Documents à fournir pendant le PFE
•
La convention de PFE, signée par les trois parties (étudiant, entreprise, ISIT) avant le
démarrage

Fin du texte :
me : exceptionnel, autorisé seulement si le sujet est assez large ; chaque étudiant
rédige alors son pro

In [5]:
# Statistiques de l'extraction

print("Nombre total de pages extraites :", len(documents))

print("\nNombre de pages par document :")

from collections import Counter

pages_par_document = Counter(
    doc["source"] for doc in documents
)

for nom, nombre in pages_par_document.items():
    print(nom, ":", nombre, "pages")

Nombre total de pages extraites : 13

Nombre de pages par document :
01_guide_pfe.pdf : 2 pages
02_reglement_stages.pdf : 3 pages
03_guide_redaction.pdf : 3 pages
04_guide_soutenance.pdf : 3 pages
05_calendrier_universitaire.pdf : 2 pages


Étape 3.2 — Nettoyage des documents

In [10]:


import re

for doc in documents:

    texte = doc["text"]

    lignes = texte.splitlines()
    lignes_nettoyees = []

    for ligne in lignes:

        # Supprimer toute ligne contenant le pied de page ISIT
        if re.search(r'ISIT\s*[–—-]', ligne):
            continue

        lignes_nettoyees.append(ligne)

    texte_nettoye = "\n".join(lignes_nettoyees)

    # Réduire les espaces multiples
    texte_nettoye = re.sub(r'[ \t]+', ' ', texte_nettoye)

    # Réduire les lignes vides multiples
    texte_nettoye = re.sub(r'\n{3,}', '\n\n', texte_nettoye)

    doc["text"] = texte_nettoye.strip()

print("Nettoyage corrigé terminé !")


Nettoyage corrigé terminé !


In [11]:
# Vérifier le texte après nettoyage

for doc in documents[:3]:

    print("Document :", doc["source"])
    print("Page :", doc["page"])
    print("\nContenu nettoyé :")
    print(doc["text"][-300:])
    print("\n" + "=" * 60)

Document : 01_guide_pfe.pdf
Page : 1

Contenu nettoyé :
lus tard le 16 novembre
2026. Il valide la démarche scientifique, relit le rapport et organise au moins 4 réunions de suivi.
•
L'encadrant professionnel est un ingénieur ou un cadre de la structure d'accueil, ayant au
moins 3 ans d'expérience. Il définit les tâches au quotidien et évalue l'étudiant.

Document : 01_guide_pfe.pdf
Page : 2

Contenu nettoyé :
, avec une assurance
internationale.
•
PFE en binôme : exceptionnel, autorisé seulement si le sujet est assez large ; chaque étudiant
rédige alors son propre rapport sur sa partie.
11. Contacts
Service des Stages et PFE : stages@isit.example.tn, bureau B-104, ouvert du lundi au vendredi de
9h à 16h.

Document : 02_reglement_stages.pdf
Page : 1

Contenu nettoyé :
0 heures par semaine, selon les horaires de l'entreprise. Le
stagiaire respecte le règlement intérieur de la structure d'accueil. Toute absence doit être justifiée
auprès de l'encadrant professionnel. Au-delà de 3 jours d

In [12]:
# Vérifier si les pieds de page ISIT sont encore présents

for doc in documents:
    if "ISIT –" in doc["text"]:
        print("Pied de page détecté :", doc["source"], "Page", doc["page"])

print("Vérification terminée.")

Vérification terminée.


Étape 3.3 — Découpage en chunks

In [13]:

# Paramètres du découpage
chunk_size = 500
chunk_overlap = 100

chunks = []

for doc in documents:

    texte = doc["text"]

    start = 0

    while start < len(texte):

        end = start + chunk_size

        chunk_text = texte[start:end]

        chunks.append({
            "text": chunk_text,
            "source": doc["source"],
            "page": doc["page"]
        })

        start += chunk_size - chunk_overlap

print("Découpage terminé !")
print("Nombre total de chunks :", len(chunks))


Découpage terminé !
Nombre total de chunks : 64
